# L2 Order Book Exploration

I am reading *Trading and Exchanges* by Larry Harris to learn about market microstructure. To me, order execution is a curious topic. Efficient order matching and execution facilitates price discovery and information aggregation: critical aspects of healthy markets.

This notebook uses Coinbase's L2 order book data and various simple statistics to explore market information regarding various cryptocurrencies.

In [16]:
import websockets
import asyncio
import json
from decimal import Decimal
import pandas as pd

In [6]:
# Coinbase WebSocket URL
coinbase_url = "wss://advanced-trade-ws.coinbase.com"

In [7]:
# WebSocket messages for BTC, ETH, XRP
messages = {
    "BTC-USD":{
        "type": "subscribe",
        "product_ids": ["BTC-USD"],
        "channel": "level2"
    },
    "ETH-USD":{
        "type": "subscribe",
        "product_ids": ["ETH-USD"],
        "channel": "level2"
    },
    "XRP-USD":{
        "type": "subscribe",
        "product_ids": ["XRP-USD"],
        "channel": "level2"
    }
}

In [22]:
async def parse_order_books():
    """
    Function for parsing L2 order book data of various cryptocurrencies
    """
    try:
        # Connect to Coinbase server
        ws = await websockets.connect(coinbase_url, max_size = 10*1024*1024)

        # Iterate through the various currencies
        for currency, message in messages.items():
            await ws.send(json.dumps(message))
            response = json.loads(await ws.recv())

            # Accessing order book according to JSON structure
            order_book = response["events"][0]["updates"]
            order_book_timestamp = response["events"][0]["updates"][0]["event_time"]

            # Obtaining the top 5 bids and top 5 asks
            # Coinbase response already sorts bids from highest to lowest and asks from lowest to highest
            top_bids = []
            top_asks = []
            for order in order_book:
                # Bids: what prices are people offering to buy a cryptocurrency in USD?
                if order["side"] == "bid" and len(top_bids) < 5:
                    top_bids.append(order)
                # Asks: what prices are people asking for to sell a cryptocurrency in USD?
                elif order["side"] == "offer" and len(top_asks) < 5:
                    top_asks.append(order)

            best_bid = Decimal(top_bids[0]["price_level"])
            best_ask = Decimal(top_asks[0]["price_level"])

            # Spread: the price difference between the lowest sell (ask) price and the highest buy (bid) price
            spread = best_ask - best_bid

            # Mid price: signals the "equilibrium price" where buyers and sellers meet
            mid_price = (best_ask + best_bid) / 2

            bid_volume = sum([Decimal(order["new_quantity"]) for order in top_bids])
            ask_volume = sum([Decimal(order["new_quantity"]) for order in top_asks])
            total_volume = bid_volume + ask_volume

            # Order book imbalance: a fraction that gives insight into how a market "feels" about an asset
            # A positive OBI means that orders are buying more than selling where a negative OBI means that orders are selling more than buying
            obi = (bid_volume - ask_volume) / total_volume

            results = {
                "currency": currency,
                "order_book_timestamp": order_book_timestamp,
                "best_bid": str(best_bid),
                "best_ask": str(best_ask),
                "spread": str(spread),
                "mid_price": str(mid_price),
                "obi": str(obi)
            }

            with open(f"{currency}_{order_book_timestamp}.txt", "w", encoding = "utf-8") as f:
                json.dump(results, f, indent = 2)

    finally:
        # Close server connection
        print("Closing connection...", end = "\r")
        await ws.close()
        print("\x1b[2KConnection closed!")

await parse_order_books()

Connection closed!
